# Model & Forecast Visualization

Use the **already-trained models** (in `models/saved/*.pkl`) to visualize two things:

- **Part A — Model evaluation**: how reliable is the model on historical data? (Actual vs Predicted on the test set)
- **Part B — Future 7-day forecast**: what does the model predict for the next 7 days? (from `predictions/*.csv`)

All plots are interactive (Plotly): hover to see values, drag to zoom.

> Tip: change `model_name` in Part A to switch between the 8 saved models.

## Part A — Model Evaluation

We want to know: **is the trained model trustworthy?**

Method (the same as in training):
1. Load the feature table `V2.5_15min_features.csv`
2. Split it chronologically (time-series split) 80/20 — train / test
3. Load a saved model (`.pkl` file)
4. Predict on the test set and compare with the real prices

### A0. Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

MODELS_DIR = Path('../models/saved')
PREDICTIONS_DIR = Path('../predictions')

print('Imports ready')

Imports ready


In [2]:
# Load the feature table and do a chronological 80/20 split (no shuffle!)
df = pd.read_csv('../data/convertData/V2.5_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)

X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_test  = X.iloc[train_end:].reset_index(drop=True)
y_test  = y.iloc[train_end:].reset_index(drop=True)
test_dt = df['datetime'].iloc[train_end:].reset_index(drop=True)

print('Test set shape:', X_test.shape)

Test set shape: (21043, 49)


In [3]:
# Load a saved model and predict on the test set
model_name = 'xgboost_v2_5'          # try also: xgboost_v1_5, lightgbm_v2_5, xgboost_v3, ...

meta = joblib.load(MODELS_DIR / f'{model_name}.pkl')
model = meta['model']
feature_cols = meta['feature_cols']
step_min = meta['step_min']
print(f'Loaded {model_name}: {len(feature_cols)} features, step {step_min} min')

# Predict ONLY on the columns the model was trained on
y_pred = model.predict(X_test[feature_cols])

# Compute evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f'Test metrics -> MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}')

Loaded xgboost_v2_5: 49 features, step 15 min
Test metrics -> MAE: 2.8231 | RMSE: 8.2191 | R2: 0.9718


### A1. Actual vs Predicted — Time Series (first 7 days of test set)

Why: the most intuitive check — plot real price and predicted price together.
If the two lines track each other closely, the model is reliable.
We show only the first 7 days of the test set so the lines are readable.

In [4]:
# 7 days * 24 hours * 4 quarters = 672 fifteen-minute steps
steps = 7 * 24 * 4

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dt[:steps], y=y_test[:steps], name='Actual', mode='lines'))
fig.add_trace(go.Scatter(x=test_dt[:steps], y=y_pred[:steps], name='Predicted', mode='lines',
                         line=dict(width=1.5)))
fig.update_layout(title=f'A1. Actual vs Predicted — First 7 Days of Test Set ({model_name})',
                  xaxis_title='Time (Europe/Helsinki)', yaxis_title='Price (EUR/MWh)')
fig.show()

### A2. Actual vs Predicted — Scatter Plot with Identity Line

Why: one point per test sample. X = actual, Y = predicted.
If the model is perfect, every point sits on the red diagonal line (y = x).
Points far above the line = the model over-predicts; far below = under-predicts.

In [5]:
# Sample points for performance (21k points is heavy for interactive plot)
rng = np.random.RandomState(42)
idx = rng.choice(len(y_test), size=min(20000, len(y_test)), replace=False)

fig = px.scatter(x=y_test.values[idx], y=y_pred[idx], opacity=0.3,
                 title=f'A2. Actual vs Predicted Scatter ({model_name})',
                 labels={'x': 'Actual Price (EUR/MWh)', 'y': 'Predicted Price (EUR/MWh)'})

# Identity line y = x (a perfect model would lie exactly on it)
lims = [float(min(y_test.min(), y_pred.min())), float(max(y_test.max(), y_pred.max()))]
fig.add_trace(go.Scatter(x=lims, y=lims, mode='lines', name='Perfect fit (y=x)',
                         line=dict(color='red', dash='dash')))
fig.show()

### A3. Residual Distribution

Why: residual (residual, 残差) = actual - predicted.
A good model has residuals centered near 0 (red dashed line) and spread symmetrically.
A wide or skewed distribution means the model struggles in some situations (e.g. price spikes).

In [6]:
residuals = y_test.values - y_pred

fig = px.histogram(residuals, nbins=80,
                   title=f'A3. Residual Distribution ({model_name})',
                   labels={'value': 'Residual = Actual - Predicted (EUR/MWh)', 'count': 'Count'})
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.show()

print(f'Residuals: mean={residuals.mean():.4f}  std={residuals.std():.4f}')

Residuals: mean=0.0950  std=8.2185


### A4. Feature Importance

Why: shows which input features (特征) the model relies on most.
This helps you understand WHAT drives the price, and which features to keep or engineer further.
For XGBoost/LightGBM this is the "gain" importance of each feature.

In [7]:
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()

fig = px.bar(imp.tail(20), orientation='h',
             title=f'A4. Top 20 Feature Importances ({model_name})',
             labels={'index': 'Feature', 'value': 'Importance'})
fig.show()

## Part B — Future 7-Day Forecast Visualization

Part A proved the model is reliable. Now the real goal: **show the next 7 days**.

The daily forecast is produced by `src/predict_system.py`, which loads the SAME `.pkl` models and saves one CSV per model into `predictions/`. Each CSV contains:
- `target_datetime` — the predicted time slot (15-min or 1-hour)
- `predicted_price` — the forecasted price
- `actual_price` — the real price (backfilled once the day has passed)

We simply read those CSVs and draw them.

In [8]:
# Load every forecast CSV produced by predict_system.py
def load_forecasts():
    frames = {}
    for path in sorted(PREDICTIONS_DIR.glob('*_forecasts.csv')):
        f = pd.read_csv(path, parse_dates=['run_date', 'target_datetime'])
        name = path.stem.replace('_forecasts', '')
        # make the time column timezone-aware (Helsinki)
        if f['target_datetime'].dt.tz is None:
            f['target_datetime'] = f['target_datetime'].dt.tz_localize('UTC').dt.tz_convert('Europe/Helsinki')
        else:
            f['target_datetime'] = f['target_datetime'].dt.tz_convert('Europe/Helsinki')
        frames[name] = f
    return frames

forecasts = load_forecasts()
print('Forecast files loaded:', len(forecasts))
for name, f in forecasts.items():
    n_runs = f['run_date'].dt.normalize().nunique()
    latest = f['run_date'].max()
    print(f'  {name}: {len(f)} rows | {n_runs} run date(s) | latest run: {latest}')

Forecast files loaded: 6
  lightgbm_v2_5: 672 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00
  lightgbm_v2: 168 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00
  xgboost_v1_5: 672 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00
  xgboost_v1: 168 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00
  xgboost_v2_5: 672 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00
  xgboost_v2: 168 rows | 1 run date(s) | latest run: 2026-08-05 15:59:29.581467+03:00


### B1. 7-Day Forecast — All Models (latest run)

Why: draw every model's most recent 7-day prediction on the same chart.
You can immediately see which models agree, and which one is an outlier.
Click the legend to show / hide a model.

In [9]:
fig = go.Figure()
for name, f in forecasts.items():
    latest = f['run_date'].max()                      # this model's most recent run
    f_latest = f[f['run_date'] == latest]
    fig.add_trace(go.Scatter(x=f_latest['target_datetime'], y=f_latest['predicted_price'],
                             name=name, mode='lines'))
fig.update_layout(title='B1. 7-Day Forecast — All Models (latest run)',
                  xaxis_title='Time (Europe/Helsinki)', yaxis_title='Predicted Price (EUR/MWh)')
fig.show()

### B2. Best Model — 7-Day Forecast vs Backfilled Actuals

Why: the best model (xgboost_v2_5) forecast, with real prices overlaid once available.
After a few days, the black dashed line (actual) appears next to the blue line (predicted),
so you can visually judge how accurate this week's forecast really was.

In [10]:
name = 'xgboost_v2_5'                # best model per README (RMSE 8.22, R2 0.972)
f = forecasts[name]
latest = f['run_date'].max()
f_latest = f[f['run_date'] == latest]

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_latest['target_datetime'], y=f_latest['predicted_price'],
                         name='Predicted', mode='lines', line=dict(width=2)))

actual = f_latest.dropna(subset=['actual_price'])
if not actual.empty:
    fig.add_trace(go.Scatter(x=actual['target_datetime'], y=actual['actual_price'],
                             name='Actual (backfilled)', mode='lines',
                             line=dict(color='black', dash='dash')))
    title = f'B2. {name} — 7-Day Forecast vs Actual'
else:
    title = f'B2. {name} — 7-Day Forecast (actuals not available yet)'

fig.update_layout(title=title,
                  xaxis_title='Time (Europe/Helsinki)', yaxis_title='Price (EUR/MWh)')
fig.show()